In [0]:
from delta.tables import DeltaTable
import os
import sys

# Ensure Python can find utils.py from the parent directory
project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from pyspark.sql import functions as F

# ==========================================
# 1. PARAMETERIZATION (Databricks Widget)
# ==========================================
dbutils.widgets.text("batch_name", "batch_1", "Target Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Processing Silver Geolocation for Batch: {batch_name}")

# ==========================================
# 2. EXTRACT INCOMING DATA
# ==========================================
df_geo_raw = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_geolocation"
).filter(F.col("_batch_id") == batch_name)

# ==========================================
# 3. TRANSFORMATIONS & CENTROID AGGREGATION
# ==========================================
# Clean individual fields first before grouping
df_prepared = df_geo_raw.select(
    F.trim(F.col("geolocation_zip_code_prefix")).alias(
        "geolocation_zip_code"
    ),
    F.col("geolocation_lat").cast("double").alias("geolocation_lat"),
    F.col("geolocation_lng").cast("double").alias("geolocation_lng"),
    utils.clean_text_column("geolocation_city").alias("geolocation_city"),
    F.upper(F.trim(F.col("geolocation_state"))).alias("geolocation_state"),
)

# Aggregate to find the central point (centroid) per zip code prefix
# This eliminates duplicates and prevents row explosion during downstream joins
df_aggregated = (
    df_prepared.groupBy("geolocation_zip_code")
    .agg(
        F.avg("geolocation_lat").alias("geolocation_lat"),
        F.avg("geolocation_lng").alias("geolocation_lng"),
        F.first("geolocation_city", ignorenulls=True).alias(
            "geolocation_city"
        ),
        F.first("geolocation_state", ignorenulls=True).alias(
            "geolocation_state"
        ),
    )
    .filter(F.col("geolocation_zip_code").isNotNull())
)

# Compute Row Hash using SHA-256 for change detection
df_hashed = df_aggregated.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("geolocation_lat").cast("string"), F.lit("")),
            F.coalesce(F.col("geolocation_lng").cast("string"), F.lit("")),
            F.coalesce(F.col("geolocation_city"), F.lit("")),
            F.coalesce(F.col("geolocation_state"), F.lit("")),
        ),
        256,
    ),
)

# ==========================================
# 4. TARGET TABLE INITIALIZATION OR UPSERT
# ==========================================
target_table_name = "ecommerce_lakehouse.silver.silver_geolocation"
table_exists = spark.catalog.tableExists(target_table_name)
current_timestamp = F.current_timestamp()

if not table_exists:
  print("Target table does not exist. Initializing silver_geolocation...")

  df_transformed = df_hashed.repartition(1).select(
      "geolocation_zip_code",
      "geolocation_lat",
      "geolocation_lng",
      "geolocation_city",
      "geolocation_state",
      "row_hash",
      current_timestamp.alias("_etl_inserted_at"),
      current_timestamp.alias("_etl_updated_at"),
      F.lit(False).alias("_delete_flag"),
  )

  df_transformed.write.format("delta").mode("overwrite").saveAsTable(
      target_table_name
  )
  print("Successfully initialized silver_geolocation table!")

else:
  print("Table exists. Executing incremental upsert merge...")

  deltaTable = DeltaTable.forName(spark, target_table_name)

  deltaTable.alias("target").merge(
      source=df_hashed.alias("source"),
      condition=(
          "target.geolocation_zip_code ="
          " source.geolocation_zip_code"
      ),
  ).whenMatchedUpdate(
      condition="target.row_hash <> source.row_hash",
      set={
          "geolocation_lat": F.col("source.geolocation_lat"),
          "geolocation_lng": F.col("source.geolocation_lng"),
          "geolocation_city": F.col("source.geolocation_city"),
          "geolocation_state": F.col("source.geolocation_state"),
          "row_hash": F.col("source.row_hash"),
          "_etl_updated_at": current_timestamp,
      },
  ).whenNotMatchedInsert(
      values={
          "geolocation_zip_code": F.col(
              "source.geolocation_zip_code"
          ),
          "geolocation_lat": F.col("source.geolocation_lat"),
          "geolocation_lng": F.col("source.geolocation_lng"),
          "geolocation_city": F.col("source.geolocation_city"),
          "geolocation_state": F.col("source.geolocation_state"),
          "row_hash": F.col("source.row_hash"),
          "_etl_inserted_at": current_timestamp,
          "_etl_updated_at": current_timestamp,
          "_delete_flag": F.lit(False),
      },
  ).execute()

  print("Silver Geolocation processing completed successfully!")

Processing Silver Geolocation for Batch: batch_1
Target table does not exist. Initializing silver_geolocation...
Successfully initialized silver_geolocation table!
